In [0]:
from pyspark.sql.functions import col, count, countDistinct, min as spark_min, max as spark_max

# =========================================================
# QUESTION 1 TESTS
# =========================================================

q1 = spark.table(
    "rearc_analytics.population_analytics.gold_population_stats"
)

print("QUESTION 1")
display(q1)

assert q1.count() == 1, "Q1 should return exactly one row."

required_q1 = {"mean_population", "stddev_population"}
assert required_q1.issubset(set(q1.columns)), \
    f"Q1 missing columns: {required_q1 - set(q1.columns)}"

row = q1.first()

assert row["mean_population"] is not None, \
    "Q1 mean_population is NULL."

assert row["stddev_population"] is not None, \
    "Q1 stddev_population is NULL."

assert row["mean_population"] > 0, \
    "Q1 mean_population should be positive."

assert row["stddev_population"] >= 0, \
    "Q1 stddev_population should not be negative."

print("✅ Question 1 passed")


# =========================================================
# QUESTION 2 TESTS
# =========================================================

q2 = spark.table(
    "rearc_analytics.population_analytics.gold_best_year_by_series"
)

print("\nQUESTION 2")
display(q2.limit(20))

required_q2 = {
    "series_id",
    "series_label",
    "best_year",
    "best_year_value"
}

assert required_q2.issubset(set(q2.columns)), \
    f"Q2 missing columns: {required_q2 - set(q2.columns)}"

assert q2.count() > 0, \
    "Q2 returned no rows."

# One Gold row per series_id
q2_counts = q2.agg(
    count("*").alias("rows"),
    countDistinct("series_id").alias("distinct_series")
).first()

assert q2_counts["rows"] == q2_counts["distinct_series"], \
    "Q2 should contain exactly one best-year row per series_id."

# Required values
assert q2.filter(col("series_id").isNull()).count() == 0, \
    "Q2 contains NULL series_id."

assert q2.filter(col("best_year").isNull()).count() == 0, \
    "Q2 contains NULL best_year."

assert q2.filter(col("best_year_value").isNull()).count() == 0, \
    "Q2 contains NULL best_year_value."

# Human-readable labels
assert q2.filter(
    col("series_label").isNull() |
    (col("series_label") == "")
).count() == 0, \
    "Q2 contains missing human-readable series labels."

print("✅ Basic Question 2 tests passed")


# =========================================================
# Q2: PROVE THE SELECTED YEAR IS REALLY THE MAX
# =========================================================

silver_bls = spark.table(
    "rearc.bls.silver_bls_data"
)

yearly = (
    silver_bls
    .filter(col("period").isin("Q01", "Q02", "Q03", "Q04"))
    .groupBy("series_id", "year")
    .sum("value")
    .withColumnRenamed("sum(value)", "annual_value")
)

actual_max = (
    yearly
    .groupBy("series_id")
    .max("annual_value")
    .withColumnRenamed("max(annual_value)", "expected_max_value")
)

q2_validation = (
    q2
    .join(actual_max, "series_id")
    .filter(
        col("best_year_value") != col("expected_max_value")
    )
)

assert q2_validation.count() == 0, \
    "Q2 contains series where best_year_value is not the maximum annual value."

print("✅ Question 2 max-year validation passed")


# =========================================================
# QUESTION 3 TESTS
# =========================================================

q3 = spark.table(
    "rearc_analytics.population_analytics.gold_series_population"
)

print("\nQUESTION 3")
display(q3.orderBy("year"))

required_q3 = {
    "year",
    "value",
    "population"
}

assert required_q3.issubset(set(q3.columns)), \
    f"Q3 missing columns: {required_q3 - set(q3.columns)}"

assert q3.count() > 0, \
    "Q3 returned no rows."

assert q3.filter(col("year").isNull()).count() == 0, \
    "Q3 contains NULL year."

assert q3.filter(col("value").isNull()).count() == 0, \
    "Q3 contains NULL BLS values."

# Compare Gold rows with the exact source filter
expected_q3 = (
    silver_bls
    .filter(
        (col("series_id") == "PRS30006032") &
        (col("period") == "Q01")
    )
    .select("year", "value")
)

assert q3.count() == expected_q3.count(), \
    "Q3 row count does not match PRS30006032 / Q01 source observations."

# Every source year/value should exist in Gold
missing_q3 = (
    expected_q3
    .join(
        q3.select("year", "value"),
        ["year", "value"],
        "left_anti"
    )
)

assert missing_q3.count() == 0, \
    "Q3 is missing one or more PRS30006032 / Q01 observations."

print("✅ Question 3 passed")


# =========================================================
# POPULATION JOIN CHECK
# =========================================================

population = spark.table(
    "rearc.datausa.silver_population"
).filter(
    col("nation") == "United States"
)

population_years = set(
    r["year"]
    for r in population.select("year").collect()
)

q3_rows = q3.select("year", "population").collect()

for r in q3_rows:
    if r["year"] in population_years:
        assert r["population"] is not None, \
            f"Population should exist for year {r['year']}."

print("✅ Population join validation passed")


print("\n🎉 ALL GOLD TESTS PASSED")

QUESTION 1


mean_population,stddev_population
3.22069808E8,3796119.936934378


✅ Question 1 passed

QUESTION 2


series_id,series_label,best_year,best_year_value
PRS30006011,Manufacturing - Employment,2022,16.400000000000002
PRS30006012,Manufacturing - Employment,2022,13.0
PRS30006013,Manufacturing - Employment,1989,578.3689999999999
PRS30006021,Manufacturing - Average weekly hours,2010,14.200000000000001
PRS30006022,Manufacturing - Average weekly hours,2010,8.9
PRS30006023,Manufacturing - Average weekly hours,2014,402.512
PRS30006031,Manufacturing - Hours worked,2022,16.5
PRS30006032,Manufacturing - Hours worked,1994,14.599999999999998
PRS30006033,Manufacturing - Hours worked,1989,568.736
PRS30006061,Manufacturing - Labor compensation,1988,28.700000000000003


✅ Basic Question 2 tests passed
✅ Question 2 max-year validation passed

QUESTION 3


year,value,population
1988,2.1,null
1989,1.8,null
1990,-4.6,null
1991,-7.9,null
1992,-3.1,null
1993,1.3,null
1994,1.7,null
1995,0.0,null
1996,-4.2,null
1997,2.8,null


✅ Question 3 passed
✅ Population join validation passed

🎉 ALL GOLD TESTS PASSED
